# Precision: aggregating the annotation campaign

Aggregates the raw exports of the labelling platform into `precision_points.csv`, one row per
annotated detection, with its coordinates, its array identifier, its unit, and a binary outcome:
was the detection real?

The notebook also serves the campaign itself. It counts how many usable points each unit has and,
where a unit falls short of the floor, draws the additional detections to annotate. Those are
written out, annotated by hand, and returned to the raw folder for the next run, which is why this
notebook is run repeatedly rather than once.

The uncertainty calculation that justifies the floor lives in `score_and_aggregation.ipynb`.


In [1]:
# Repository bootstrap: locate the root, then read every data location from paths.py.
import sys
from pathlib import Path


def _repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for parent in [start, *start.parents]:
        if (parent / "paths.py").exists() and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("run this notebook from inside the repository")


sys.path.insert(0, str(_repo_root()))
import paths


## Retrieval of the raw annotations, aggregation and indexation

### Retrieval and aggregation

In [2]:
import os
import glob
import pandas as pd
import geopandas as gpd

folder = paths.ANNOTATIONS_RAW / "precision"

files = sorted(glob.glob(os.path.join(folder, "*.geojson")))
print(f"{len(files)} files found in {folder}/")

gdfs = [gpd.read_file(f) for f in files]
labels_precision = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
print(f"{len(labels_precision)} features before filtering")

# Drop points not yet reviewed
labels_precision = labels_precision[labels_precision["reviewTag"] != "unreviewed"].reset_index(drop=True)
print(f"{len(labels_precision)} features after dropping the unreviewed")

# Deduplicate on geometry: one installation can appear in several files when
# the municipal extractions overlap.
labels_precision["_geom_key"] = labels_precision.geometry.apply(lambda g: g.wkb)
labels_precision = labels_precision.drop_duplicates(subset="_geom_key").drop(columns="_geom_key").reset_index(drop=True)
print(f"{len(labels_precision)} features after deduplication")

labels_precision.drop(columns=["array_id", "rnb_id", 'dpt_source'], axis=1, inplace=True)
labels_precision.head()

27 files found in /sessions/happy-gracious-franklin/mnt/Publication/source/data/source/annotations/raw/precision/


52260 features before filtering
42410 features after dropping the unreviewed
27316 features after deduplication


/sessions/happy-gracious-franklin/tmp/ipykernel_10/2453188109.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  labels_precision = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)


,insee,nom,population,dpt,surface,tilt,azimuth,kWp,year,reviewTag,reviewTimestamp,index_right,geometry
0,75056,Paris,2113705.0,75,30.810,23.44,4.09,4.74,2024,normal,2025-11-20 13:09:52.406000+00:00,NaN,"POLYGON ((2.25414 48.84391, 2.25414 48.84389, ..."
1,75056,Paris,2113705.0,75,47.840,20.00,-58.00,7.36,2024,false,2025-11-20 13:16:04.975000+00:00,NaN,"POLYGON ((2.25721 48.86313, 2.25721 48.86311, ..."
2,75056,Paris,2113705.0,75,13.585,23.44,-84.31,2.09,2024,false,2025-11-20 13:14:42.644000+00:00,NaN,"POLYGON ((2.25891 48.84376, 2.25891 48.84374, ..."
3,75056,Paris,2113705.0,75,116.610,20.00,-56.00,17.94,2024,normal,2025-11-20 13:08:36.266000+00:00,NaN,"POLYGON ((2.26058 48.84939, 2.26058 48.84937, ..."
4,75056,Paris,2113705.0,75,21.125,20.00,2.86,3.25,2024,false,2025-11-20 13:16:37.237000+00:00,NaN,"POLYGON ((2.2607 48.85635, 2.2607 48.85633, 2...."


### Unit-level patches

Any review file placed in the patches folder **replaces every annotation of its unit**: the base
annotations of the units a patch covers are dropped and the patch takes their place.

This exists for one purpose: fully re-labelling a unit whose agreement score came out too low to
trust. Paris was the case that motivated it, at 0.435 before re-labelling. Patches follow the same
conventions as the base annotations.

In [3]:
import glob as _glob

PATCHES_DIR = os.path.join(folder, 'patches')  # unit-level replacements, if any
patch_files = sorted(_glob.glob(os.path.join(PATCHES_DIR, '*.geojson')))
if patch_files:
    patches = gpd.GeoDataFrame(
        pd.concat([gpd.read_file(f) for f in patch_files], ignore_index=True),
        crs=labels_precision.crs)
    patches = patches[patches['reviewTag'] != 'unreviewed'].reset_index(drop=True)
    # Review files can carry leftover columns from earlier spatial joins
    patches = patches.drop(columns=[c for c in ('index_right', 'index_left') if c in patches.columns])
    # Which units the patches cover: a fast spatial join on the centroids
    deps = gpd.read_file(paths.DEPARTEMENTS)[['code', 'geometry']]
    cent = patches.copy(); cent['geometry'] = cent.geometry.centroid
    j = gpd.sjoin(cent, deps, how='left', predicate='within')
    j = j[~j.index.duplicated(keep='first')]   # a centroid on a boundary joins twice
    patched_depts = sorted(set(j['code'].dropna()))
    # Drop the base annotations of those units, then add the patches
    cent_base = labels_precision.drop(
        columns=[c for c in ('index_right', 'index_left') if c in labels_precision.columns]).copy()
    cent_base['geometry'] = cent_base.geometry.centroid
    jb = gpd.sjoin(cent_base, deps, how='left', predicate='within')
    jb = jb[~jb.index.duplicated(keep='first')]   # same
    keep_mask = ~jb['code'].isin(patched_depts)
    n_dropped = (~keep_mask).sum()
    labels_precision = pd.concat([labels_precision[keep_mask.values], patches],
                                  ignore_index=True)
    labels_precision = gpd.GeoDataFrame(labels_precision, crs=patches.crs)
    print(f"PATCHES: {len(patch_files)} file(s), units replaced: {patched_depts} "
          f"({n_dropped} base annotations dropped, {len(patches)} patch annotations added)")
else:
    print(f"no patch in {PATCHES_DIR}/")


PATCHES: 1 file(s), units replaced: ['75', '93'] (857 base annotations dropped, 157 patch annotations added)


/sessions/happy-gracious-franklin/tmp/ipykernel_10/249266504.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = patches.copy(); cent['geometry'] = cent.geometry.centroid
/sessions/happy-gracious-franklin/tmp/ipykernel_10/249266504.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent_base['geometry'] = cent_base.geometry.centroid


### Recovering each annotation's array identifier

Some export files already carry an identifier taken from the detections. The older ones, hand-drawn
squares, never had one and always depended on a spatial join to find the corresponding detection.

**The choice made here** is to ignore any identifier already present and keep only the geometry, so
that the spatial join is uniform across every file, old and new.

That was checked rather than assumed: it gives the same result as preferring the known identifier
where one exists. The unmatched count is identical either way, and only four rows out of 8,260 in
the newest file differ, all cases of adjacent detected polygons that touch. The gain is uniform
behaviour and no column collision in the join, at no cost in accuracy.

In [4]:
import numpy as np
import geopandas as gpd

detections = gpd.read_file(
    paths.DETECTIONS
)

# Align the coordinate systems
if labels_precision.crs != detections.crs:
    detections = detections.to_crs(labels_precision.crs)

# Clean copy
labels_precision_id = labels_precision.copy()
# A leftover identifier, from a patch file attached to an older detection
# release, would corrupt the merge below. The spatial join against the CURRENT
# product is what counts.
labels_precision_id = labels_precision_id.drop(
    columns=[c for c in ('array_id',) if c in labels_precision_id.columns])

# Unique label identifier
labels_precision_id["label_id"] = labels_precision_id.index

# Spatial join
matches = gpd.sjoin(
    labels_precision_id[["label_id", "geometry"]],
    detections[["array_id", "geometry"]],
    how="left",
    predicate="intersects",
)

# One match per label
matches = (
    matches.groupby("label_id", as_index=False)["array_id"]
    .first()
)

# Merge the result back
labels_precision_id = labels_precision_id.merge(
    matches,
    on="label_id",
    how="left",
)

labels_precision_id = labels_precision_id.drop(columns="label_id")
labels_precision_id = labels_precision_id[labels_precision_id['array_id'].notna()]

print(
    f"{len(labels_precision_id)} labels matched"
)

labels_precision_id.head()

18201 labels matched


,insee,nom,population,dpt,surface,tilt,azimuth,kWp,year,reviewTag,reviewTimestamp,index_right,geometry,rnb_id,dpt_source,array_id
20,50359,Mortain-Bocage,2954.0,50,77.30,30.94,45.88,7.69,2019,false,2025-11-20 14:37:38.314000+00:00,NaN,"POLYGON ((-0.90136 48.59622, -0.90136 48.5962,...",NaN,NaN,339478.0
28,50046,Bérigny,442.0,50,19.44,45.42,-3.01,3.09,2019,normal,2025-11-20 14:38:02.393000+00:00,NaN,"POLYGON ((-0.93755 49.1443, -0.93755 49.14428,...",NaN,NaN,337802.0
35,50444,Saint-Amand-Villages,2523.0,50,14.79,45.42,-14.93,2.35,2019,normal,2025-11-20 09:56:02.644000+00:00,NaN,"POLYGON ((-0.94942 49.05828, -0.94942 49.05826...",NaN,NaN,340034.0
42,50444,Saint-Amand-Villages,2523.0,50,26.35,27.92,26.57,3.28,2019,normal,2025-11-20 09:39:22.742000+00:00,NaN,"POLYGON ((-0.95546 49.0417, -0.95546 49.04168,...",NaN,NaN,340021.0
43,50450,Saint-Barthélemy,338.0,50,25.17,30.94,0.00,3.13,2019,normal,2025-11-20 09:55:57.659000+00:00,NaN,"POLYGON ((-0.95566 48.67731, -0.95566 48.67729...",NaN,NaN,340071.0


## Counting points per unit

In [5]:
import pandas as pd

# Tag categories
positive_tags = ["true", "normal"]
negative_tags = ["false"]
unknown_tags = ["unknown"]

# Indicator columns
labels_precision_id["is_positive"] = labels_precision_id["reviewTag"].isin(positive_tags)
labels_precision_id["is_negative"] = labels_precision_id["reviewTag"].isin(negative_tags)
labels_precision_id["is_unknown"] = labels_precision_id["reviewTag"].isin(unknown_tags)

# Aggregate per unit
stats_dpt = (
    labels_precision_id.groupby("dpt")
      .agg(
          n_annotations=("reviewTag", "size"),
          n_positive=("is_positive", "sum"),
          n_negative=("is_negative", "sum"),
          n_unknown=("is_unknown", "sum"),
      )
)

# Usable sample size
stats_dpt["n_valid_samples"] = (
    stats_dpt["n_positive"] + stats_dpt["n_negative"]
)

# Precision
stats_dpt["precision"] = (
    stats_dpt["n_positive"] / stats_dpt["n_valid_samples"]
)

# Units carrying only "unknown" tags
stats_dpt["precision"] = stats_dpt["precision"].fillna(0)

### Setting the per-unit annotation floor

In [6]:

# Default floor
DEFAULT_MIN_SAMPLES = 120

# Optional per-unit override
min_samples = {
     #"75": 1000,
    # "13": 200,
    # "974": 50,
}

# Per-unit constraint
stats_dpt["min_samples_required"] = (
    stats_dpt.index.map(min_samples)
    .fillna(DEFAULT_MIN_SAMPLES)
    .astype(int)
)

# Does the unit meet its floor?
stats_dpt["has_enough_samples"] = (
    stats_dpt["n_valid_samples"] >= stats_dpt["min_samples_required"]
)

stats_dpt[stats_dpt['has_enough_samples']==False]

,n_annotations,n_positive,n_negative,n_unknown,n_valid_samples,precision,min_samples_required,has_enough_samples
dpt,,,,,,,,
74,120,106,11,3,117,0.905983,120,False
93,1,0,0,1,0,0.000000,120,False


### Drawing the next batch to annotate

In [7]:
SEED = 42

# Which units need more sampling. The floor is evaluated at the REPORTING UNIT
# level, not the department: the members of the merged Paris unit pool their
# annotations, so an apparent shortfall in one member does not trigger sampling
# when the unit's pool already clears the floor.
import sys as _sys
_sys.path.insert(0, os.path.abspath('../../../..'))
import units as _units
_valid_dpt = labels_precision_id['dpt'].astype(str) if 'dpt' in labels_precision_id.columns else None
_unit_counts = _units.to_unit(stats_dpt.index.astype(str).to_series(index=stats_dpt.index))
_pool = stats_dpt.groupby(_unit_counts.values)['n_valid_samples'].sum()
_floor = stats_dpt['min_samples_required'].iloc[0]
dpt_to_sample = stats_dpt.index.astype(str)[
    (stats_dpt["has_enough_samples"] == False).values
    & (_pool.reindex(_unit_counts.values).values < _floor)
]

# Arrays already annotated
selected_samples = (
    labels_precision_id["array_id"]
    .astype(int)
    .tolist()
)

# Remaining pool
remaining_samples = detections[
    ~detections["array_id"].isin(selected_samples)
].copy()

# Normalise the unit code
remaining_samples["dpt"] = remaining_samples["dpt"].astype(str)
dpt_to_sample = dpt_to_sample.astype(str)

samples = []

for dpt in dpt_to_sample:

    # How many are needed to reach the floor
    n_needed = int(
        stats_dpt.loc[dpt, "min_samples_required"]
        - stats_dpt.loc[dpt, "n_valid_samples"]
    )

    if n_needed <= 0:
        continue

    # Pool available in that unit
    pool_dpt = remaining_samples[
        remaining_samples["dpt"] == dpt
    ]

    if len(pool_dpt) == 0:
        print(f"unit {dpt}: no sample available")
        continue

    # Never ask for more than exists
    n_sample = min(n_needed, len(pool_dpt))

    if n_sample < n_needed:
        print(
            f"unit {dpt}: needs {n_needed}, "
            f"available {len(pool_dpt)} -> {n_sample} taken"
        )

    samples.append(
        pool_dpt.sample(
            n=n_sample,
            random_state=SEED
        )
    )


# Final aggregation
if samples:
    to_label = gpd.GeoDataFrame(
        pd.concat(samples, ignore_index=True),
        crs=detections.crs
    )
else:
    to_label = gpd.GeoDataFrame(
        columns=detections.columns,
        crs=detections.crs
    )

print(f"{len(to_label)} new points to annotate")

# Written through a temporary file: a direct overwrite is refused on some
# filesystems.
import tempfile, shutil as _sh, os as _os
_tmp = _os.path.join(tempfile.mkdtemp(), 'to_label.geojson')
to_label.to_file(_tmp, driver="GeoJSON")
_sh.copy(_tmp, "to_label.geojson")

3 new points to annotate


'to_label.geojson'

In [8]:
check = (
    to_label.groupby("dpt")
    .size()
    .rename("sampled")
    .to_frame()
    .join(
        stats_dpt[
            ["n_valid_samples", "min_samples_required"]
        ],
        how="left"
    )
)

check["expected"] = (
    check["min_samples_required"]
    - check["n_valid_samples"]
)



check['expected'].sum()

np.int64(3)

## Output: `precision_points.csv`

One row per labelled annotation, with the array identifier, coordinates, unit, and outcome.

The outcome is 1 for a confirmed true positive, 0 for a confirmed false positive. Undecided
annotations are excluded rather than assigned, since assigning them would invent a judgement the
annotator declined to make.

One point about what this column is not. Because every annotated object is already a detection, the
model has no continuous score left to threshold here: the outcome encodes the human verdict on the
model's output, not a model prediction. That is exactly the quantity precision needs.


In [9]:
TP_TAGS = ["true", "normal"]
FP_TAGS = ["false"]

# On a duplicate array, the latest review wins. Rare, around 0.1% of rows.
n_before_dedup = len(labels_precision_id)
labels_precision_id = (
    labels_precision_id.sort_values('reviewTimestamp')
    .drop_duplicates(subset='array_id', keep='last')
)
print(f"{n_before_dedup} -> {len(labels_precision_id)} after deduplicating on the array (latest review kept)")

pts = labels_precision_id.copy()
pts['point'] = pts.geometry.representative_point()

pred_map = {**{t: 1 for t in TP_TAGS}, **{t: 0 for t in FP_TAGS}}
pts['pred'] = pts['reviewTag'].map(pred_map)

n_unknown = pts['pred'].isna().sum()
pts = pts[pts['pred'].notna()].copy()
print(f"{n_unknown} undecided annotations excluded")

precision_points = pd.DataFrame({
    'array_id': pts['array_id'].astype(int),
    'lat': pts['point'].y,
    'lon': pts['point'].x,
    'dpt': pts['dpt'],
    'pred': pts['pred'].astype(int),
})

precision_points.to_csv(paths.PRECISION_POINTS, index=False)

print(f"\nwrote precision_points.csv ({len(precision_points)} rows)")
print(precision_points['pred'].value_counts())
precision_points.head()


18201 -> 18178 after deduplicating on the array (latest review kept)
167 undecided annotations excluded



wrote precision_points.csv (18011 rows)
pred
1    13802
0     4209
Name: count, dtype: int64


,array_id,lat,lon,dpt,pred
9839,233721,48.245520,-2.141330,35,1
9845,231249,47.933245,-1.661111,35,1
9848,232116,48.319277,-1.211322,35,1
9851,230983,48.125989,-1.368441,35,1
9852,231216,48.078688,-1.310654,35,1
